# Fraud Detection Model Training with Ray

This notebook demonstrates distributed XGBoost training using Ray for fraud detection.

In [ ]:
import os
import ray
import boto3
import pandas as pd
import numpy as np
from ray.train.xgboost import XGBoostTrainer
from ray.train import ScalingConfig
from ray import tune
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

## Environment Setup

In [ ]:
# Get environment variables
RAY_ADDRESS = os.environ.get('RAY_ADDRESS', 'ray://ray-cluster-head:10001')
S3_BUCKET = os.environ.get('S3_BUCKET')
AWS_REGION = os.environ.get('AWS_DEFAULT_REGION', 'us-west-2')

print(f"Ray Address: {RAY_ADDRESS}")
print(f"S3 Bucket: {S3_BUCKET}")
print(f"Region: {AWS_REGION}")

## Connect to Ray Cluster

In [ ]:
# Connect to Ray cluster
try:
    ray.init(address=RAY_ADDRESS, ignore_reinit_error=True)
    print("Connected to Ray cluster")
    print(f"Ray cluster resources: {ray.cluster_resources()}")
except Exception as e:
    print(f"Failed to connect to Ray cluster: {e}")
    print("Starting local Ray instance...")
    ray.init(ignore_reinit_error=True)

## Load Data from S3

In [ ]:
def load_fraud_data_from_s3():
    """
    Load processed fraud detection features from S3
    """
    s3_path = f"s3://{S3_BUCKET}/fraud-data/processed-features/"
    
    try:
        # Use Ray Data to load from S3
        import ray.data as rd
        dataset = rd.read_parquet(s3_path)
        df = dataset.to_pandas()
        print(f"Loaded {len(df)} records from S3")
        return df
    except Exception as e:
        print(f"Failed to load from S3: {e}")
        print("Using sample data for demonstration...")
        return create_sample_data()

def create_sample_data(n_samples=10000):
    """
    Create sample fraud detection data
    """
    np.random.seed(42)
    
    # Generate features
    data = {
        'tx_amount': np.random.lognormal(3, 1, n_samples),
        'hour': np.random.randint(0, 24, n_samples),
        'day_of_week': np.random.randint(1, 8, n_samples),
        'customer_tx_count_15min': np.random.poisson(2, n_samples),
        'customer_avg_amount_15min': np.random.lognormal(3, 0.5, n_samples),
        'terminal_risk_score': np.random.beta(2, 5, n_samples),
        'customer_age_days': np.random.randint(1, 3650, n_samples),
    }
    
    # Generate fraud labels (imbalanced)
    fraud_probability = (
        0.01 +  # Base fraud rate
        0.05 * (data['tx_amount'] > np.percentile(data['tx_amount'], 95)) +  # High amounts
        0.03 * (data['hour'] < 6) +  # Late night
        0.02 * (data['customer_tx_count_15min'] > 5)  # High frequency
    )
    
    data['tx_fraud'] = np.random.binomial(1, fraud_probability, n_samples)
    
    df = pd.DataFrame(data)
    print(f"Created sample dataset with {len(df)} records")
    print(f"Fraud rate: {df['tx_fraud'].mean():.3f}")
    
    return df

# Load data
df = load_fraud_data_from_s3()
df.head()

## Data Preparation

In [ ]:
# Prepare features and target
feature_columns = [col for col in df.columns if col != 'tx_fraud']
X = df[feature_columns]
y = df['tx_fraud']

print(f"Features: {feature_columns}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {y.value_counts()}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## Distributed XGBoost Training with Ray

In [ ]:
# Convert to Ray datasets
import ray.data as rd

train_dataset = rd.from_pandas(pd.concat([X_train, y_train], axis=1))
test_dataset = rd.from_pandas(pd.concat([X_test, y_test], axis=1))

print(f"Ray train dataset: {train_dataset.count()} rows")
print(f"Ray test dataset: {test_dataset.count()} rows")

In [ ]:
# Configure XGBoost trainer
trainer = XGBoostTrainer(
    scaling_config=ScalingConfig(
        num_workers=2,  # Adjust based on cluster size
        use_gpu=False,  # Set to True if GPU workers available
        resources_per_worker={"CPU": 2}
    ),
    label_column="tx_fraud",
    datasets={"train": train_dataset, "valid": test_dataset},
    params={
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "auc"],
        "max_depth": 6,
        "learning_rate": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": (len(y_train) - y_train.sum()) / y_train.sum(),  # Handle imbalance
        "random_state": 42
    },
    num_boost_round=100,
)

print("Starting distributed XGBoost training...")
result = trainer.fit()
print("Training completed!")

## Model Evaluation

In [ ]:
# Get the trained model
checkpoint = result.checkpoint
model = XGBoostTrainer.get_model(checkpoint)

# Make predictions
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba > 0.5).astype(int)

# Evaluate model
auc_score = roc_auc_score(y_test, y_pred_proba)
print(f"AUC Score: {auc_score:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## Hyperparameter Tuning with Ray Tune

In [ ]:
# Hyperparameter tuning (optional)
def tune_xgboost():
    """
    Hyperparameter tuning with Ray Tune
    """
    tuner = tune.Tuner(
        XGBoostTrainer,
        param_space={
            "scaling_config": ScalingConfig(
                num_workers=2,
                use_gpu=False,
                resources_per_worker={"CPU": 2}
            ),
            "label_column": "tx_fraud",
            "datasets": {"train": train_dataset, "valid": test_dataset},
            "params": {
                "objective": "binary:logistic",
                "eval_metric": ["logloss", "auc"],
                "max_depth": tune.randint(3, 10),
                "learning_rate": tune.loguniform(0.01, 0.3),
                "subsample": tune.uniform(0.6, 1.0),
                "colsample_bytree": tune.uniform(0.6, 1.0),
                "scale_pos_weight": (len(y_train) - y_train.sum()) / y_train.sum(),
                "random_state": 42
            },
            "num_boost_round": 100,
        },
        tune_config=tune.TuneConfig(
            metric="valid-auc",
            mode="max",
            num_samples=5,  # Number of trials
        ),
    )
    
    results = tuner.fit()
    best_result = results.get_best_result()
    
    print(f"Best AUC: {best_result.metrics['valid-auc']:.4f}")
    print(f"Best parameters: {best_result.config['params']}")
    
    return best_result

# Uncomment to run hyperparameter tuning
# best_result = tune_xgboost()

## Save Model to S3

In [ ]:
def save_model_to_s3(model, model_name="fraud_detection_xgboost"):
    """
    Save trained model to S3
    """
    import tempfile
    import joblib
    
    # Save model locally first
    with tempfile.TemporaryDirectory() as temp_dir:
        model_path = f"{temp_dir}/{model_name}.joblib"
        joblib.dump(model, model_path)
        
        # Upload to S3
        s3_client = boto3.client('s3')
        s3_key = f"fraud-models/{model_name}.joblib"
        
        s3_client.upload_file(model_path, S3_BUCKET, s3_key)
        
        s3_uri = f"s3://{S3_BUCKET}/{s3_key}"
        print(f"Model saved to: {s3_uri}")
        
        return s3_uri

# Save the model
model_uri = save_model_to_s3(model)
print(f"Model URI: {model_uri}")

## Deploy Model with Ray Serve (Optional)

In [ ]:
from ray import serve
import joblib

@serve.deployment(num_replicas=2, ray_actor_options={"num_cpus": 1})
class FraudDetectionModel:
    def __init__(self, model_uri):
        # In production, load from S3
        self.model = model  # Use the trained model
        
    async def __call__(self, request):
        # Parse request
        features = request.json()
        
        # Convert to DataFrame
        import pandas as pd
        df = pd.DataFrame([features])
        
        # Make prediction
        fraud_probability = self.model.predict_proba(df)[0, 1]
        
        return {
            "fraud_probability": float(fraud_probability),
            "is_fraud": bool(fraud_probability > 0.5),
            "model_version": "1.0"
        }

# Deploy the model (uncomment to deploy)
# serve.start()
# FraudDetectionModel.deploy(model_uri)
# print("Model deployed with Ray Serve!")
# print("Test endpoint: http://localhost:8000/FraudDetectionModel")

## Next Steps

1. **Production Deployment**: Use Ray Serve for scalable model serving
2. **Model Monitoring**: Set up monitoring for model performance
3. **A/B Testing**: Use Ray Serve for model A/B testing
4. **Pipeline Integration**: Integrate with Argo Workflows for automated retraining

In [ ]:
# Clean up
ray.shutdown()